# Séance 11 – Fichiers et modules de base
### Lab : fichiers, pathlib, math, random, datetime, time et re

| | |
|---|---|
| **Durée estimée** | 60 min (2 × 30 min) + bonus |
| **Projet fil rouge** | GestParc – étape 8 : garanties, rapport texte, analyse du journal de maintenance |

**Objectifs**
- Lire et écrire des fichiers texte avec with, les bons modes et l'encodage
- Manipuler des chemins et des dossiers avec pathlib
- Utiliser math, random, datetime et time
- Valider et extraire des données avec des expressions régulières

**Mode d'emploi**
- Exécuter une cellule : **Maj + Entrée** ; en cas de comportement étrange : *Redémarrer le noyau et tout exécuter*.
- Complétez les cellules marquées `# TODO`, puis exécutez la cellule **✅ Vérification** qui suit : aucune erreur = réponse correcte.
- Difficulté : ⭐ facile · ⭐⭐ intermédiaire · ⭐⭐⭐ avancé · Indices dans les blocs 💡 repliables.

> ℹ️ Tous les fichiers sont créés dans le dossier `lab_s12/`, à côté du notebook. Vous pouvez le supprimer à tout moment.

In [ ]:
# Cellule de préparation (à exécuter en premier)
from pathlib import Path

WORK = Path("lab_s12")
WORK.mkdir(exist_ok=True)
print("Dossier de travail :", WORK.resolve())

---
## Partie 1 – Fichiers et chemins

> **Rappel**
> - `with open(chemin, "r" | "w" | "a" | "x", encoding="utf-8") as f:` → fermeture garantie
> - Lecture : `f.read()`, `f.readlines()`, `for line in f:` · écriture : `f.write(texte)` (pas de `\n` automatique)
> - `Path("a") / "b.txt"`, `.exists()`, `.mkdir(parents=True, exist_ok=True)`, `.glob("*.log")`, `.read_text()`, `.write_text()`, `.stat().st_size`, `.rename()`

### Exemple guidé : écrire puis analyser un petit fichier CSV « à la main »

In [ ]:
inventory_file = WORK / "postes.txt"
with open(inventory_file, "w", encoding="utf-8") as f:
    f.write("LT-001;Dell;2023\n")
    f.write("LT-002;Lenovo;2024\n")
    f.write("LT-003;HP;2019\n")

old = []
with open(inventory_file, encoding="utf-8") as f:
    for line in f:
        tag, brand, year = line.strip().split(";")
        if int(year) <= 2020:
            old.append(tag)
print(inventory_file.read_text(encoding="utf-8"), old, inventory_file.stat().st_size, "octets")

### Exercice 1 ⭐ – Écrire puis relire une liste de serveurs

1. Écrivez la fonction `save_lines(path, lines)` qui écrit chaque élément de `lines` sur une ligne (encodage UTF-8, fichier écrasé s'il existe).
2. Écrivez `load_lines(path) -> list[str]` qui renvoie les lignes **sans** le saut de ligne final et **sans** les lignes vides.

In [ ]:
def save_lines(path: Path, lines: list[str]) -> None:
    """Écrit une ligne par élément (fichier écrasé)."""
    # TODO


def load_lines(path: Path) -> list[str]:
    """Relit les lignes non vides d'un fichier."""
    # TODO


servers_file = WORK / "servers.txt"
save_lines(servers_file, ["srv-web-01", "srv-db-01", "", "srv-app-01"])
print(load_lines(servers_file))

<details><summary>💡 Indice</summary>

`f.write()` n'ajoute pas de saut de ligne. Un fichier ouvert se parcourt avec `for line in f`.

</details>

In [ ]:
# ✅ Vérification
p = WORK / "test_ex1.txt"
save_lines(p, ["a", "", "b"])
assert p.read_text(encoding="utf-8") == "a\n\nb\n"
assert load_lines(p) == ["a", "b"]
save_lines(p, ["c"])
assert load_lines(p) == ["c"], "le fichier doit être écrasé"
print("✅ Exercice 1 réussi")

### Exercice 2 ⭐⭐ – Lire un fichier de configuration

Écrivez `read_config(path) -> dict[str, str]` pour un fichier au format `clé = valeur` :
- lignes vides et commentaires (`#` en début de ligne, ou après la valeur) ignorés ;
- espaces autour des clés et valeurs supprimés ; la valeur peut contenir des `=` (seul le premier sépare) ;
- une ligne sans `=` lève `ValueError("ligne 4 invalide : ...")`.

In [ ]:
def read_config(path: Path) -> dict[str, str]:
    """Lit un fichier clé = valeur."""
    # TODO


config_file = WORK / "app.conf"
config_file.write_text("""# Configuration de l'application
db_host = srv-db-01
db_port=5432   # port PostgreSQL

connection_string = host=srv-db-01 user=app
log_level = INFO
""", encoding="utf-8")
print(read_config(config_file))

<details><summary>💡 Indice</summary>

`raw.split("#", 1)[0]` retire un commentaire de fin de ligne ; `line.split("=", 1)` ne coupe qu'au premier `=`.

</details>

In [ ]:
# ✅ Vérification
assert read_config(config_file) == {"db_host": "srv-db-01", "db_port": "5432",
                                    "connection_string": "host=srv-db-01 user=app", "log_level": "INFO"}
bad = WORK / "bad.conf"
bad.write_text("a = 1\n\n# c\nsans_egal\n", encoding="utf-8")
try:
    read_config(bad)
    raise AssertionError("ValueError attendue")
except ValueError as exc:
    assert "ligne 4" in str(exc)
print("✅ Exercice 2 réussi")

### Exercice 3 ⭐⭐ – Journal en mode ajout et dernières lignes

1. `log_event(path, timestamp, message)` **ajoute** une ligne `"2026-10-21 08:15:00 | message"` à la fin du fichier (créé s'il n'existe pas).
2. `tail(path, n=3) -> list[str]` renvoie les `n` dernières lignes (sans `\n`), ou toutes si le fichier en contient moins.

In [ ]:
from datetime import datetime


def log_event(path: Path, timestamp: datetime, message: str) -> None:
    """Ajoute un événement horodaté à la fin du journal."""
    # TODO


def tail(path: Path, n: int = 3) -> list[str]:
    """Dernières lignes d'un fichier."""
    # TODO


journal = WORK / "deploy.log"
journal.unlink(missing_ok=True)
for minute, message in enumerate(["build", "tests", "déploiement", "contrôle santé", "terminé"]):
    log_event(journal, datetime(2026, 10, 21, 8, minute), message)
print(tail(journal), tail(journal, 10)[0])

<details><summary>💡 Indice</summary>

Mode `"a"` pour ajouter. Une f-string accepte un format de date : `f"{ts:%Y-%m-%d %H:%M:%S}"`. `lines[-n:]` donne les n derniers éléments.

</details>

In [ ]:
# ✅ Vérification
assert tail(journal) == ["2026-10-21 08:02:00 | déploiement", "2026-10-21 08:03:00 | contrôle santé", "2026-10-21 08:04:00 | terminé"]
assert len(tail(journal, 10)) == 5 and tail(journal, 1) == ["2026-10-21 08:04:00 | terminé"]
log_event(journal, datetime(2026, 10, 21, 9, 0), "rollback")
assert len(journal.read_text(encoding="utf-8").splitlines()) == 6
print("✅ Exercice 3 réussi")

### Exercice 4 ⭐⭐ – Rotation de journaux avec pathlib

Le dossier `lab_s12/logs/` contient des journaux `app.log`, `app.log.1`, `app.log.2`… Écrivez `rotate(log_dir, name="app.log", keep=3)`, qui conserve au plus `keep` archives :
1. supprime `app.log.<keep>` s'il existe (l'archive la plus ancienne) ;
2. renomme `app.log.(i)` en `app.log.(i+1)` pour i de `keep - 1` à 1 (en partant du plus ancien) ;
3. renomme `app.log` en `app.log.1` et crée un nouveau `app.log` vide ;
4. renvoie la liste triée des noms de fichiers du dossier.

Testez avec `keep=2`.

Calculez ensuite `total_size`, la taille totale (octets) des fichiers `app.log*`.

In [ ]:
def rotate(log_dir: Path, name: str = "app.log", keep: int = 3) -> list[str]:
    """Rotation de journaux : app.log -> app.log.1 -> app.log.2 ..."""
    # TODO


log_dir = WORK / "logs"
log_dir.mkdir(exist_ok=True)
for p in log_dir.iterdir():
    p.unlink()
(log_dir / "app.log").write_text("jour 3\n" * 10, encoding="utf-8")
(log_dir / "app.log.1").write_text("jour 2\n" * 5, encoding="utf-8")
(log_dir / "app.log.2").write_text("jour 1\n", encoding="utf-8")
print(rotate(log_dir, keep=2))
total_size = ...

<details><summary>💡 Indice</summary>

`range(keep - 1, 0, -1)` parcourt 2, 1. `Path.unlink(missing_ok=True)` ne lève pas d'erreur si le fichier est absent.

</details>

In [ ]:
# ✅ Vérification
assert sorted(p.name for p in log_dir.iterdir()) == ["app.log", "app.log.1", "app.log.2"]
assert (log_dir / "app.log").read_text(encoding="utf-8") == ""
assert (log_dir / "app.log.1").read_text(encoding="utf-8").startswith("jour 3")
assert (log_dir / "app.log.2").read_text(encoding="utf-8").startswith("jour 2")
assert total_size == 7 * 15
print("✅ Exercice 4 réussi")

### Exercice 5 ⭐⭐ – Problèmes d'encodage

Un export d'un vieux logiciel Windows (`export_cp1252.txt`) est encodé en `cp1252`. Écrivez `read_text_any(path) -> tuple[str, str]` qui
essaie successivement les encodages `utf-8`, puis `cp1252`, et renvoie `(texte, encodage_utilisé)`. Si aucun ne fonctionne, laissez remonter l'exception.

In [ ]:
def read_text_any(path: Path, encodings: tuple[str, ...] = ("utf-8", "cp1252")) -> tuple[str, str]:
    """Lit un fichier texte en essayant plusieurs encodages."""
    # TODO


legacy = WORK / "export_cp1252.txt"
legacy.write_text("Matériel réformé – coût résiduel : 120 DT\n", encoding="cp1252")
modern = WORK / "export_utf8.txt"
modern.write_text("Équipe réseau – Sfax\n", encoding="utf-8")
print(read_text_any(legacy), read_text_any(modern))

<details><summary>💡 Indice</summary>

`path.read_text(encoding=...)` lève `UnicodeDecodeError` si les octets ne correspondent pas à l'encodage.

</details>

In [ ]:
# ✅ Vérification
assert read_text_any(legacy) == ("Matériel réformé – coût résiduel : 120 DT\n", "cp1252")
assert read_text_any(modern) == ("Équipe réseau – Sfax\n", "utf-8")
print("✅ Exercice 5 réussi")

---
## Partie 2 – Modules math, random, datetime, time et re

> **Rappel**
> - `import math` · `from datetime import date, datetime, timedelta` · `import re`
> - `math.ceil`, `math.floor`, `math.log2`, `math.sqrt` · `random.seed`, `randint`, `choice`, `sample`, `shuffle`
> - `datetime(…) + timedelta(hours=8)`, `d2 - d1` → `timedelta`, `.total_seconds()`, `strftime` / `strptime`, `fromisoformat`, `weekday()` (lundi = 0)
> - `re.fullmatch`, `re.search`, `re.findall`, `re.sub`, `re.compile`, groupes `(...)` et `(?P<nom>...)`

### Exemple guidé : horodatage et durée d'une sauvegarde

In [ ]:
import math
from datetime import date, datetime, timedelta

start = datetime(2026, 10, 21, 1, 0)
size_gb, throughput_mb_s = 420, 180
duration = timedelta(seconds=size_gb * 1024 / throughput_mb_s)
end = start + duration
print(f"Fin prévue : {end:%d/%m/%Y %H:%M} (durée {duration})")
print("Bandes LTO-9 de 18 To nécessaires pour 50 To :", math.ceil(50 / 18))
print("Jour de la semaine :", ["lun", "mar", "mer", "jeu", "ven", "sam", "dim"][date(2026, 10, 21).weekday()])

### Exercice 6 ⭐ – Dimensionner une salle serveur

Avec le module `math`, calculez :
- `racks` : nombre de baies de 42 U nécessaires pour `servers` serveurs de 2 U chacun, en gardant 6 U libres par baie ;
- `subnet_bits` : nombre de bits d'hôte nécessaires pour adresser `hosts` machines (+ 2 adresses réservées), soit `ceil(log2(hosts + 2))` ;
- `prefix` : le préfixe CIDR correspondant (`32 - subnet_bits`).

In [ ]:
import math

servers, hosts = 55, 300
# TODO
racks = ...
subnet_bits = ...
prefix = ...

<details><summary>💡 Indice</summary>

110 U nécessaires pour 36 U utiles par baie → 3,05… → 4 baies.

</details>

In [ ]:
# ✅ Vérification
assert racks == 4 and subnet_bits == 9 and prefix == 23
print("✅ Exercice 6 réussi")

### Exercice 7 ⭐⭐ – Tirage au sort reproductible pour un audit

Pour un audit de sécurité, on tire au sort des postes à contrôler. Écrivez `audit_sample(assets, ratio, seed) -> list[str]` qui :
- fixe la graine avec `random.seed(seed)` ;
- tire **sans remise** `ceil(len(assets) * ratio)` postes avec `random.sample` ;
- renvoie la liste **triée**.

Le même appel doit toujours donner le même résultat ; une autre graine, en général un autre.

In [ ]:
import random


def audit_sample(assets: list[str], ratio: float, seed: int) -> list[str]:
    """Échantillon reproductible de postes à auditer."""
    # TODO


assets = [f"LT-{n:03d}" for n in range(1, 121)]
print(audit_sample(assets, 0.05, seed=2026))

<details><summary>💡 Indice</summary>

`random.sample(population, k)` tire k éléments distincts. Pour un vrai mot de passe, on utiliserait `secrets`.

</details>

In [ ]:
# ✅ Vérification
s1 = audit_sample(assets, 0.05, seed=2026)
assert len(s1) == 6 and len(set(s1)) == 6 and set(s1) <= set(assets) and s1 == sorted(s1)
assert audit_sample(assets, 0.05, seed=2026) == s1, "même graine, même tirage"
assert audit_sample(assets, 0.05, seed=7) != s1
assert len(audit_sample(assets, 0.01, seed=1)) == 2
print("✅ Exercice 7 réussi")

### Exercice 8 ⭐⭐ – Délais de résolution des tickets

Pour chaque ticket (ouverture et résolution au format `JJ/MM/AAAA HH:MM`, priorité), calculez :
- `durations` : numéro → durée de résolution en heures (arrondie à 1 décimale) ;
- `breaches` : numéros des tickets dont la durée dépasse le SLA de leur priorité (`SLA_HOURS`) ;
- `weekend_openings` : numéros des tickets ouverts un samedi ou un dimanche ;
- `report` : `"INC-0101 résolu le mercredi 21/10 à 10h45"` pour le premier ticket (jours en français, à partir de `DAYS`).

In [ ]:
from datetime import datetime

SLA_HOURS = {"P1": 4, "P2": 8, "P3": 24}
DAYS = ["lundi", "mardi", "mercredi", "jeudi", "vendredi", "samedi", "dimanche"]
tickets = [
    ("INC-0101", "21/10/2026 08:30", "21/10/2026 10:45", "P1"),
    ("INC-0102", "17/10/2026 22:10", "19/10/2026 09:00", "P3"),
    ("INC-0103", "20/10/2026 14:00", "21/10/2026 09:30", "P2"),
    ("INC-0104", "18/10/2026 07:00", "18/10/2026 13:20", "P2"),
]
FMT = "%d/%m/%Y %H:%M"
# TODO
durations, breaches, weekend_openings = {}, [], []
report = ...

<details><summary>💡 Indice</summary>

`datetime.strptime(texte, FMT)` ; `(fin - début).total_seconds() / 3600` ; `weekday()` vaut 5 le samedi et 6 le dimanche.

</details>

In [ ]:
# ✅ Vérification
assert durations == {"INC-0101": 2.2, "INC-0102": 34.8, "INC-0103": 19.5, "INC-0104": 6.3}
assert breaches == ["INC-0102", "INC-0103"]
assert weekend_openings == ["INC-0102", "INC-0104"]
assert report == "INC-0101 résolu le mercredi 21/10 à 10h45"
print("✅ Exercice 8 réussi")

### Exercice 9 ⭐ – Chronométrer

Mesurez avec `time.perf_counter()` le temps de 2 000 recherches d'un élément **absent** dans une liste de 50 000 adresses, puis dans un ensemble contenant les mêmes adresses.
Stockez les durées dans `list_time` et `set_time`, et `speedup = list_time / set_time`. Que constatez-vous ?

In [ ]:
import time

addresses = [f"10.{i // 65536}.{i // 256 % 256}.{i % 256}" for i in range(50_000)]
address_set = set(addresses)
# TODO
list_time = ...
set_time = ...
speedup = ...

<details><summary>💡 Indice</summary>

Une recherche dans une liste parcourt les éléments un par un ; un ensemble utilise une table de hachage.

</details>

In [ ]:
# ✅ Vérification
assert list_time > 0 and set_time > 0
assert speedup > 10, "l'ensemble devrait être beaucoup plus rapide"
print("✅ Exercice 9 réussi")

### Exercice 10 ⭐⭐⭐ – Validations et extractions avec re

Écrivez :
1. `is_valid_email(text)` : `prenom.nom@domaine.tn` ; partie locale en lettres minuscules, chiffres, `.`, `_`, `-` ; domaine en lettres minuscules, chiffres, `-` et `.` ; extension de 2 à 6 lettres ;
2. `extract_ips(text)` : toutes les adresses IPv4 (format `a.b.c.d`, sans contrôler les valeurs) dans l'ordre d'apparition ;
3. `mask_card_numbers(text)` : remplace chaque groupe de 16 chiffres (éventuellement séparés par des espaces ou des tirets par blocs de 4) par `**** **** **** 1234` (4 derniers chiffres conservés) ;
4. `parse_ticket_ref(text)` : renvoie `(type, numéro_entier)` pour la **première** référence `INC-1234` ou `REQ-1234` trouvée, ou `None`.

In [ ]:
import re


def is_valid_email(text: str) -> bool:
    # TODO
    ...


def extract_ips(text: str) -> list[str]:
    # TODO
    ...


def mask_card_numbers(text: str) -> str:
    # TODO
    ...


def parse_ticket_ref(text: str) -> tuple[str, int] | None:
    # TODO
    ...


print(is_valid_email("amira.ben-salah@entreprise.tn"), extract_ips("deny 10.0.0.5 -> 192.168.1.20:443"))
print(mask_card_numbers("Paiement carte 4970 1234 5678 9012 accepté"), parse_ticket_ref("Re: [REQ-0042] accès VPN"))

<details><summary>💡 Indice</summary>

- `fullmatch` pour valider une chaîne entière ; `findall` pour extraire ; `sub` avec `\1` pour réutiliser un groupe.
- `(?:...)` est un groupe **non capturant** : avec `findall`, seuls les groupes capturants sont renvoyés.
- `\b` marque une frontière de mot.

</details>

In [ ]:
# ✅ Vérification
assert is_valid_email("amira.ben-salah@entreprise.tn") and is_valid_email("k_t2@dsi.groupe.com")
for bad in ["Amira@entreprise.tn", "amira@", "amira entreprise.tn", "a@b.c", "a@b.tn.", "@x.tn"]:
    assert not is_valid_email(bad), bad
assert extract_ips("deny 10.0.0.5 -> 192.168.1.20:443 (version 1.2.3)") == ["10.0.0.5", "192.168.1.20"]
assert mask_card_numbers("cb 4970-1234-5678-9012 ou 4970123456789012") == "cb **** **** **** 9012 ou **** **** **** 9012"
assert parse_ticket_ref("Re: [REQ-0042] accès VPN, voir INC-0007") == ("REQ", 42)
assert parse_ticket_ref("pas de référence, INC-12") is None
print("✅ Exercice 10 réussi")

---
### Exercice de synthèse ⭐⭐⭐ – Analyse d'un journal d'accès web

Le fichier `access.log` (créé ci-dessous) suit le format Apache simplifié :
`IP - - [21/Oct/2026:08:15:02 +0100] "GET /intranet/ HTTP/1.1" 200 5120 0.042`
(IP, horodatage, méthode, URL, protocole, code HTTP, taille en octets, durée en secondes).

Écrivez `analyze_access_log(path) -> dict` qui renvoie :
- `"requests"` : nombre de lignes valides ; `"invalid"` : nombre de lignes non conformes ;
- `"by_status_class"` : `{"2xx": …, "4xx": …, "5xx": …}` (classes présentes uniquement) ;
- `"top_ip"` : `(ip, nombre)` de l'IP la plus active ;
- `"slowest"` : `(url, durée)` de la requête la plus lente ;
- `"by_hour"` : heure (entier) → nombre de requêtes ;
- `"total_mb"` : volume total en Mo (1 Mo = 1 048 576 octets), arrondi à 2 décimales.

Écrivez enfin le rapport `lab_s12/access_report.txt` (une ligne par clé, format libre).

In [ ]:
import re


def analyze_access_log(path: Path) -> dict:
    """Statistiques d'un journal d'accès web."""
    # TODO


access_log = WORK / "access.log"
access_log.write_text("""10.0.4.17 - - [21/Oct/2026:08:15:02 +0100] "GET /intranet/ HTTP/1.1" 200 5120 0.042
10.0.4.17 - - [21/Oct/2026:08:15:09 +0100] "GET /intranet/annuaire HTTP/1.1" 200 20480 0.180
10.0.7.3 - - [21/Oct/2026:08:47:55 +0100] "POST /api/tickets HTTP/1.1" 201 512 0.950
10.0.4.17 - - [21/Oct/2026:09:02:13 +0100] "GET /intranet/rapport.pdf HTTP/1.1" 200 2097152 1.730
192.168.1.20 - - [21/Oct/2026:09:05:40 +0100] "GET /admin HTTP/1.1" 403 128 0.003
ligne tronquée par une rotation de journal
10.0.7.3 - - [21/Oct/2026:09:30:01 +0100] "GET /api/inventaire HTTP/1.1" 500 64 2.410
10.0.9.8 - - [21/Oct/2026:10:00:00 +0100] "GET /intranet/ HTTP/1.1" 304 0 0.010
""", encoding="utf-8")
stats = analyze_access_log(access_log)
# TODO : écrire lab_s12/access_report.txt

<details><summary>💡 Indice</summary>

- Mettez au point le motif sur une seule ligne avant de l'appliquer au fichier (regex101.com aide beaucoup).
- `\S+` : une suite de caractères non blancs (l'URL). Les guillemets du journal sont dans le motif : chaîne délimitée par `'...'`.

</details>

In [ ]:
# ✅ Vérification
assert stats["requests"] == 7 and stats["invalid"] == 1
assert stats["by_status_class"] == {"2xx": 4, "4xx": 1, "5xx": 1, "3xx": 1}
assert stats["top_ip"] == ("10.0.4.17", 3)
assert stats["slowest"] == ("/api/inventaire", 2.41)
assert stats["by_hour"] == {8: 3, 9: 3, 10: 1}
assert stats["total_mb"] == 2.03
assert (WORK / "access_report.txt").exists()
print("✅ Exercice de synthèse réussi")

---
## Projet fil rouge – étape 8 : garanties, rapport texte, journal de maintenance

Le modèle de l'étape 7 a été **enrichi** (fourni dans `scripts/gestparc/gestparc_modele.py`) :
- chaque équipement a une **date d'achat** `purchase_date` (propriété, modifiable) ;
- chaque classe définit sa durée de garantie `WARRANTY_MONTHS` (portable 36 mois, serveur 60, imprimante 12, réseau 60) ;
- propriétés et méthodes `warranty_end`, `warranty_days_left(today)`, `is_under_warranty(today)`, fonction `add_months`.

La cellule suivante charge ce modèle dans le notebook avec la commande Jupyter `%run` (les imports de modules seront vus en S14).

**À faire :**
1. `expiring_warranties(inventory, within_days, today)` → liste de `(numéro, jours restants)` pour les garanties qui expirent dans l'horizon (expirées exclues), triée ;
2. `write_inventory_report(inventory, path, today)` → rapport texte UTF-8 (titre daté, une ligne par équipement trié avec le statut de garantie, ligne de total) ; renvoie le nombre de lignes ;
3. `LOG_PATTERN` et `parse_maintenance_log(path)` → `(entrées, lignes_invalides)` ; chaque entrée est le `groupdict()` du motif, où `date` et `time` sont remplacés par une clé `timestamp` (`datetime`) ;
4. `log_summary(entries, inventory)` → `{"by_level": …, "alerts_by_asset": … (WARN et ERROR), "unknown_assets": … (liste triée des numéros absents de l'inventaire)}`.

In [ ]:
%run scripts/gestparc/gestparc_modele.py
from datetime import date, datetime
import re

TODAY = date(2026, 10, 21)
inventory = demo_inventory()
for equipment in sorted(inventory):
    print(equipment.asset_tag, equipment.purchase_date, "→ garantie jusqu'au", equipment.warranty_end)

In [ ]:
LOG_PATTERN = re.compile(
    r"TODO"   # groupes nommés : date, time, level, asset, message
)


def expiring_warranties(inventory: Inventory, within_days: int, today: date) -> list[tuple[str, int]]:
    """Garanties qui expirent dans les within_days jours (déjà expirées exclues), de la plus proche à la plus lointaine."""
    # TODO
    ...


def write_inventory_report(inventory: Inventory, path: Path, today: date) -> int:
    """Écrit un rapport texte UTF-8 de l'inventaire ; renvoie le nombre de lignes écrites."""
    # TODO
    ...


def parse_maintenance_log(path: Path) -> tuple[list[dict], list[int]]:
    """Analyse un journal de maintenance ; renvoie (entrées valides, numéros des lignes invalides)."""
    # TODO
    ...


def log_summary(entries: list[dict], inventory: Inventory) -> dict:
    """Synthèse du journal : niveaux, alertes par équipement, équipements inconnus."""
    # TODO
    ...


print(expiring_warranties(inventory, 120, TODAY))
lines = write_inventory_report(inventory, WORK / "rapport_inventaire.txt", TODAY)
print((WORK / "rapport_inventaire.txt").read_text(encoding="utf-8"))
(WORK / "maintenance.log").write_text(Path("scripts/gestparc/sample_maintenance.log").read_text(encoding="utf-8"), encoding="utf-8")
entries, invalid = parse_maintenance_log(WORK / "maintenance.log")
print(len(entries), "entrées ; lignes invalides :", invalid)
print(log_summary(entries, inventory))

In [ ]:
# ✅ Vérification
assert expiring_warranties(inventory, 120, TODAY) == [("LT-001", 58), ("LT-002", 93)]
assert expiring_warranties(inventory, 30, TODAY) == []
report_lines = (WORK / "rapport_inventaire.txt").read_text(encoding="utf-8").splitlines()
assert lines == len(report_lines) == 9
assert report_lines[0] == "Inventaire GestParc – 21/10/2026"
assert report_lines[2].startswith("LT-001") and "sous garantie jusqu'au 18/12/2026" in report_lines[2]
assert "hors garantie" in report_lines[4] and report_lines[-1].startswith("5 équipements")
assert invalid == [7, 11] and len(entries) == 8
assert entries[0]["timestamp"] == datetime(2026, 10, 19, 7, 58, 10) and entries[0]["asset"] == "SV-001"
assert "date" not in entries[0] and entries[2]["message"] == "bourrage papier bac 2"
summary = log_summary(entries, inventory)
assert summary["by_level"] == {"INFO": 3, "WARN": 3, "ERROR": 2}
assert summary["alerts_by_asset"] == {"SV-001": 2, "PR-001": 1, "SW-001": 1, "SV-009": 1}
assert summary["unknown_assets"] == ["SV-009"]
print("✅ Étape 8 du projet GestParc validée")

✔️ Version de référence : `scripts/gestparc/main.py` (menu : garanties à échéance, export du rapport, analyse d'un journal). Lancez-la depuis le dossier `scripts/gestparc/`.

**Prochaine étape (S13)** : import / export CSV et JSON, et persistance dans une base **SQLite**.

---
## Pour aller plus loin (bonus)

1. **`secrets`** : générez un mot de passe temporaire de 16 caractères (lettres, chiffres, symboles) avec `secrets.choice`.
2. **`timeit`** : comparez précisément `"x" in liste` et `"x" in ensemble` avec `timeit.timeit(..., number=1000)`.
3. **`shutil`** : archivez le dossier `lab_s12/logs` en `.zip` avec `shutil.make_archive`.
4. **`zoneinfo`** : affichez l'heure courante à Tunis, Paris et Montréal (`datetime.now(ZoneInfo("Africa/Tunis"))`).

In [ ]:
# À vous